# Problem Set 2, Problem 4: Computation graphs and automatic differentiation
ROB-GY 6013 Mathematics for Robotics, Fall 2026

**Instructions.** Write code only where a cell says `TODO`, and run all cells from top to bottom. Do not use PyTorch or any other automatic-differentiation library.

**Background (optional).** The engine below follows Andrej Karpathy's micrograd. The first video of his series *Neural Networks: Zero to Hero* builds it step by step: https://karpathy.ai/zero-to-hero.html

In [ ]:
import math
import random
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator

## Part (a): the `Value` class

A `Value` stores a number `data` and a number `grad`, and it remembers the `Value`s it was computed from. Arithmetic with `Value`s therefore records a computation graph. After `L.backward()`, every `Value` `v` that `L` depends on has `v.grad` equal to $\partial L/\partial v$.

Every operation stores a small function `_backward`. It adds the operation's local derivatives, multiplied by the gradient `out.grad` of its output, to the `grad` of its inputs. `backward()` calls these functions from `L` back to the inputs, so each call is one application of the chain rule.

Addition and `backward()` are complete. Complete the two TODOs below, `_backward` for multiplication and for ReLU, following the example of addition.

In [ ]:
class Value:
    """A single number that remembers how it was computed."""

    def __init__(self, data, children=()):
        self.data = float(data)      # the number itself
        self.grad = 0.0              # dL/d(this Value), filled in by backward()
        self._children = children    # the Values this one was computed from
        self._backward = lambda: None

    def __repr__(self):
        return f"Value(data={self.data:.6g}, grad={self.grad:.6g})"

    # Addition, given as an example. For out = self + other,
    # d(out)/d(self) = 1 and d(out)/d(other) = 1.
    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other))
        def _backward():
            self.grad += 1.0 * out.grad
            other.grad += 1.0 * out.grad
        out._backward = _backward
        return out

    # Multiplication: out = self * other.
    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other))
        def _backward():
            # TODO (a): add the local derivatives, multiplied by out.grad,
            # to self.grad and other.grad.
            raise NotImplementedError("part (a): multiplication")
        out._backward = _backward
        return out

    # ReLU: out = max(0, self). Take the derivative at 0 to be 0.
    def relu(self):
        out = Value(max(0.0, self.data), (self,))
        def _backward():
            # TODO (a): add the local derivative, multiplied by out.grad, to self.grad.
            raise NotImplementedError("part (a): ReLU")
        out._backward = _backward
        return out

    # Subtraction and arithmetic with plain numbers, built from + and *.
    def __radd__(self, other):
        return self + other

    def __rmul__(self, other):
        return self * other

    def __neg__(self):
        return self * -1.0

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):
        return other + (-self)

    def backward(self):
        """Fill in v.grad = dL/dv for every Value v that this Value L depends on."""
        # Order the graph so that every Value comes after the Values it was computed from.
        order, visited = [], set()
        def visit(v):
            if v not in visited:
                visited.add(v)
                for child in v._children:
                    visit(child)
                order.append(v)
        visit(self)
        # dL/dL = 1. Then go from L back to the inputs. Each _backward call
        # is one application of the chain rule.
        self.grad = 1.0
        for v in reversed(order):
            v._backward()

### Test for part (a)

Run this cell after completing the two TODOs. It should print `-3.0 2.0`, then `0.0 1.0`, then `6.0`.

In [ ]:
# Test for part (a). Expected output:
#   -3.0 2.0
#   0.0 1.0
#   6.0
a = Value(2.0)
b = Value(-3.0)
L = a * b + 1
L.backward()
print(a.grad, b.grad)

c = Value(-1.5)
d = Value(2.0)
L = c.relu() + d.relu()
L.backward()
print(c.grad, d.grad)

u = Value(3.0)
L = u * u      # dL/du = 2u = 6
L.backward()
print(u.grad)

### What happens in `L = a * b + 1`

Run this cell after part (a). It prints each `Value` with the `Value`s it was computed from, before and after `backward()`.

In [ ]:
a = Value(3.0)
b = Value(4.0)

# Forward: each operation creates a new Value and remembers its inputs.
t = a * b      # t.data = 12, computed from (a, b)
L = t + 1      # L.data = 13, computed from (t, 1)
print("t =", t, "computed from", t._children)
print("L =", L, "computed from", L._children)

# Backward: set L.grad = 1, then call _backward from L back to the inputs.
# L's _backward fills in t.grad; only then does t's _backward fill in a.grad and b.grad.
L.backward()
print("after backward:")
print("L =", L)
print("t =", t)
print("a =", a)
print("b =", b)

## Helper functions

Nothing to change here. A network is a list of layers, the loss is $L=\frac{1}{2n}\sum_{k=1}^{n}(\hat y_k-y_k)^2$, and `zero_grad(net)` resets every gradient to zero.

In [ ]:
# A network is a list of layers (W, b) of Values. W[j][i] connects unit i of one
# layer to unit j of the next, and b[j] is the bias of unit j. ReLU follows every
# layer except the last.

def make_network(sizes, seed):
    """Random network with the given layer sizes, for example [1, 16, 1]."""
    rng = random.Random(seed)
    net = []
    for n_in, n_out in zip(sizes[:-1], sizes[1:]):
        w = [[rng.gauss(0.0, math.sqrt(1.0 / n_in)) for j in range(n_out)] for i in range(n_in)]
        W = [[Value(w[i][j]) for i in range(n_in)] for j in range(n_out)]
        b = [Value(0.0) for j in range(n_out)]
        net.append((W, b))
    return net

def forward(net, x):
    """Output of the network for one input x, given as a list of numbers."""
    a = x
    for k, (W, b) in enumerate(net):
        z = [b[j] + sum(a[i] * W[j][i] for i in range(len(a))) for j in range(len(b))]
        a = [zj.relu() for zj in z] if k < len(net) - 1 else z
    return a

def loss_of(net, xs, ys):
    """L = (1/2n) * sum over the n examples of (yhat - y)^2, as a Value."""
    total = 0.0
    for x, y in zip(xs, ys):
        e = forward(net, x)[0] - y
        total = total + 0.5 * e * e
    return total * (1.0 / len(xs))

def parameters(net):
    """All weights and biases of the network, as one list of Values."""
    return [p for W, b in net for row in W for p in row] + [p for W, b in net for p in b]

def zero_grad(net):
    """Reset the gradient of every parameter to zero."""
    for p in parameters(net):
        p.grad = 0.0

def random_data(n, d, seed):
    """n random examples with d inputs each, and random targets."""
    rng = random.Random(seed)
    xs = [[rng.gauss(0.0, 1.0) for i in range(d)] for k in range(n)]
    ys = [rng.gauss(0.0, 1.0) for k in range(n)]
    return xs, ys

def plot_loss(history, step_size):
    plt.figure(figsize=(5, 3))
    plt.semilogy(history, "o-" if len(history) <= 20 else "-")
    plt.gca().xaxis.set_major_locator(MaxNLocator(integer=True))
    plt.xlabel("step")
    plt.ylabel("loss")
    plt.title(f"Training loss, step size {step_size}")
    plt.savefig(f"p4-loss-{step_size}.png", dpi=150, bbox_inches="tight")
    plt.show()

def plot_fit(net, xs, ys):
    grid = [-math.pi + 2 * math.pi * k / 199 for k in range(200)]
    plt.figure(figsize=(5, 3))
    plt.plot(grid, [math.sin(t) for t in grid], label="sin x")
    plt.plot(grid, [forward(net, [t])[0].data for t in grid], label="network")
    plt.scatter([x[0] for x in xs], ys, s=12, color="black", label="samples")
    plt.legend()
    plt.title("Fit after training")
    plt.savefig("p4-fit.png", dpi=150, bbox_inches="tight")
    plt.show()

## Part (b): a two-layer network

Write the forward pass of the Problem 3 network with `Value`s, call `backward()` once, and compare the printed gradients with your answer to Problem 3(c).

In [ ]:
# The network and training example of Problem 3.
x = [2.0, 1.0]
y = 1.0
W1 = [[Value(1.0), Value(-1.0)],
      [Value(-1.0), Value(1.0)]]    # W1[j][i] connects input i to hidden unit j
b1 = [Value(1.0), Value(0.0)]
W2 = [[Value(2.0), Value(1.0)]]     # W2[0][j] connects hidden unit j to the output
b2 = Value(-1.0)

# TODO (b): compute z1, z2, a1, a2, yhat, e = yhat - y and L = 0.5 * e * e one line
# at a time, using only Value arithmetic and .relu(). Then call L.backward().


print("L      =", L.data)
print("dL/dW1 =", [[w.grad for w in row] for row in W1])
print("dL/db1 =", [v.grad for v in b1])
print("dL/dW2 =", [[w.grad for w in row] for row in W2])
print("dL/db2 =", b2.grad)

## Part (c): finite-difference check

For each parameter `p`, compare `p.grad` with the centred difference $\dfrac{L(p+\varepsilon)-L(p-\varepsilon)}{2\varepsilon}$ with $\varepsilon=10^{-4}$, using the scaled error $\dfrac{|a-b|}{\max(1,\,|a|,\,|b|)}$. The largest error should be below $10^{-8}$.

In [ ]:
net = make_network([3, 4, 1], seed=0)
xs, ys = random_data(n=5, d=3, seed=1)
eps = 1e-4
errors = []

# TODO (c): call backward() once on loss_of(net, xs, ys) to fill in every p.grad.
# Then, for each p in parameters(net):
#   - evaluate the loss with p.data increased by eps, and with p.data decreased by eps,
#     using loss_of(net, xs, ys).data,
#   - restore p.data,
#   - compute estimate = (loss_plus - loss_minus) / (2 * eps),
#   - append |p.grad - estimate| / max(1, |p.grad|, |estimate|) to errors.


print("parameters checked:", len(errors))
print("largest scaled error:", max(errors))

## Part (d): gradient descent

Train a network with layer sizes 1, 16, 1 on 20 samples of $\sin x$ by gradient descent with step size $0.1$ for 1000 steps. The final loss should be below $0.01$. Then, in the next cell, train a fresh copy with step size $0.5$ for 5 steps. The plotting functions save the figures as `p4-loss-0.1.png`, `p4-fit.png` and `p4-loss-0.5.png`.

In [ ]:
xs = [[-math.pi + 2 * math.pi * k / 19] for k in range(20)]
ys = [math.sin(x[0]) for x in xs]
net = make_network([1, 16, 1], seed=0)
lr = 0.1
history = []

# TODO (d): 1000 steps of gradient descent. In every step: call zero_grad(net),
# compute L = loss_of(net, xs, ys), call L.backward(), update every parameter
# with p.data -= lr * p.grad, and append L.data to history.


print("final loss:", loss_of(net, xs, ys).data)
plot_loss(history, 0.1)
plot_fit(net, xs, ys)

In [ ]:
net = make_network([1, 16, 1], seed=0)
lr = 0.5
history = []

# TODO (d): the same loop for 5 steps with lr = 0.5, appending L.data to history.


print("losses:", history)
plot_loss(history, 0.5)